In [ ]:
# ==========================================
# TimeGAN Implementation for Bitcoin Data
# Project: BDA Comprehensive Evaluation Project (CEP)
# Author: [Muhammad Fahim, IBrar Ullah, Abdul Wasey, Rozi Khan]
# ==========================================

# --- 1. SETUP & MOUNT ---
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses
from sklearn.preprocessing import MinMaxScaler
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# Define Paths
BASE_PATH = '/content/drive/MyDrive/BDA_CEP_TimeGAN'
DATA_DIR = os.path.join(BASE_PATH, 'data')
MODEL_DIR = os.path.join(BASE_PATH, 'models')
OUTPUT_DIR = os.path.join(BASE_PATH, 'outputs')

# Create directories if they don't exist
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Environment Setup Complete.")

# --- 2. DATA LOADING & PROCESSING ---
file_path = os.path.join(BASE_PATH, 'bitcoin.csv') # Verify this path matches your Drive

if os.path.exists(file_path):
    print("Reading file...")
    df = pd.read_csv(file_path)

    # Preprocessing
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
    df = df.set_index('Timestamp')
    df_daily = df.resample('D').mean().ffill().bfill()

    # Feature Selection
    vol_col = [c for c in df_daily.columns if "Volume" in c][0]
    features = ['Open', 'High', 'Low', 'Close', vol_col]
    data_values = df_daily[features].values

    # Scaling
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data_values)

    # Sequence Creation
    SEQ_LEN = 24
    data_seq = []
    for i in range(len(data_scaled) - SEQ_LEN):
        data_seq.append(data_scaled[i : i + SEQ_LEN])
    data_seq = np.array(data_seq)

    # Shuffle for training
    np.random.shuffle(data_seq)

    print(f"✅ Data Loaded & Processed. Shape: {data_seq.shape}")
else:
    raise FileNotFoundError(f"File not found at {file_path}")

# --- 3. MODEL ARCHITECTURE ---
HIDDEN_DIM = 24
BATCH_SIZE = 128
N_FEATURES = 5

def make_embedder():
    return models.Sequential([
        layers.Input(shape=(SEQ_LEN, N_FEATURES)),
        layers.GRU(HIDDEN_DIM, return_sequences=True),
        layers.GRU(HIDDEN_DIM, return_sequences=True),
        layers.Dense(HIDDEN_DIM, activation='sigmoid')
    ], name="Embedder")

def make_recovery():
    return models.Sequential([
        layers.Input(shape=(SEQ_LEN, HIDDEN_DIM)),
        layers.GRU(HIDDEN_DIM, return_sequences=True),
        layers.GRU(HIDDEN_DIM, return_sequences=True),
        layers.Dense(N_FEATURES, activation='sigmoid')
    ], name="Recovery")

def make_generator():
    return models.Sequential([
        layers.Input(shape=(SEQ_LEN, HIDDEN_DIM)),
        layers.GRU(HIDDEN_DIM, return_sequences=True),
        layers.GRU(HIDDEN_DIM, return_sequences=True),
        layers.Dense(HIDDEN_DIM, activation='sigmoid')
    ], name="Generator")

def make_discriminator():
    return models.Sequential([
        layers.Input(shape=(SEQ_LEN, HIDDEN_DIM)),
        layers.GRU(HIDDEN_DIM, return_sequences=True),
        layers.GRU(HIDDEN_DIM, return_sequences=True),
        layers.Dense(1, activation=None)
    ], name="Discriminator")

def make_supervisor():
    return models.Sequential([
        layers.Input(shape=(SEQ_LEN, HIDDEN_DIM)),
        layers.GRU(HIDDEN_DIM, return_sequences=True),
        layers.Dense(HIDDEN_DIM, activation='sigmoid')
    ], name="Supervisor")

# Initialize Models
embedder = make_embedder()
recovery = make_recovery()
generator = make_generator()
discriminator = make_discriminator()
supervisor = make_supervisor()

# Optimizers
emb_opt = tf.keras.optimizers.Adam(learning_rate=0.001)
rec_opt = tf.keras.optimizers.Adam(learning_rate=0.001)
gen_opt = tf.keras.optimizers.Adam(learning_rate=0.001)
disc_opt = tf.keras.optimizers.Adam(learning_rate=0.001)

print("✅ Models Initialized.")

# --- 4. TRAINING LOOPS ---
@tf.function
def train_autoencoder(x):
    with tf.GradientTape() as tape:
        h = embedder(x)
        x_tilde = recovery(h)
        loss = losses.MeanSquaredError()(x, x_tilde)
    vars = embedder.trainable_variables + recovery.trainable_variables
    grads = tape.gradient(loss, vars)
    emb_opt.apply_gradients(zip(grads, vars))
    return loss

@tf.function
def train_supervisor(x):
    with tf.GradientTape() as tape:
        h = embedder(x)
        h_hat_sup = supervisor(h)
        loss = losses.MeanSquaredError()(h[:, 1:, :], h_hat_sup[:, :-1, :])
    vars = supervisor.trainable_variables + generator.trainable_variables
    grads = tape.gradient(loss, vars)
    gen_opt.apply_gradients(zip(grads, vars))
    return loss

@tf.function
def train_generator(x, z):
    with tf.GradientTape() as tape:
        h_hat = generator(z)
        h_hat_sup = supervisor(h_hat)
        y_fake = discriminator(h_hat_sup)

        loss_u = losses.BinaryCrossentropy(from_logits=True)(tf.ones_like(y_fake), y_fake)
        loss_s = losses.MeanSquaredError()(h_hat[:, 1:, :], h_hat_sup[:, :-1, :])
        x_hat = recovery(h_hat_sup)
        loss_m = losses.MeanSquaredError()(tf.reduce_mean(x, 0), tf.reduce_mean(x_hat, 0)) + \
                 losses.MeanSquaredError()(tf.math.reduce_std(x, 0), tf.math.reduce_std(x_hat, 0))

        g_loss = loss_u + 10 * loss_s + 10 * loss_m
    vars = generator.trainable_variables + supervisor.trainable_variables
    grads = tape.gradient(g_loss, vars)
    gen_opt.apply_gradients(zip(grads, vars))
    return g_loss

@tf.function
def train_discriminator(x, z):
    with tf.GradientTape() as tape:
        h = embedder(x)
        h_hat = generator(z)
        h_hat_sup = supervisor(h_hat)
        y_real = discriminator(h)
        y_fake = discriminator(h_hat_sup)
        d_loss = losses.BinaryCrossentropy(from_logits=True)(tf.ones_like(y_real), y_real) + \
                 losses.BinaryCrossentropy(from_logits=True)(tf.zeros_like(y_fake), y_fake)
    vars = discriminator.trainable_variables
    grads = tape.gradient(d_loss, vars)
    disc_opt.apply_gradients(zip(grads, vars))
    return d_loss

# Start Training
train_data = tf.data.Dataset.from_tensor_slices(data_seq).shuffle(1000).batch(BATCH_SIZE)
EPOCHS = 100 # Adjusted for quick demo, increase for final result

print("🚀 Starting Training...")
# Phase 1: AE
for e in range(EPOCHS):
    for x in train_data: train_autoencoder(x)
print("Phase 1 Complete.")

# Phase 2: Supervisor
for e in range(EPOCHS):
    for x in train_data: train_supervisor(x)
print("Phase 2 Complete.")

# Phase 3: Joint
for e in range(EPOCHS):
    for x in train_data:
        z = tf.random.normal(shape=(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM))
        train_generator(x, z)
        train_discriminator(x, z)
print("✅ Training Complete.")

# --- 5. DATA GENERATION (INFERENCE) ---
print("Generating Synthetic Data...")
Z_noise = tf.random.normal(shape=(len(data_seq), SEQ_LEN, HIDDEN_DIM))
generated_latent = supervisor(generator(Z_noise))
generated_data = recovery(generated_latent).numpy()

# Prepare flattened arrays for visualization
real_flattened = data_seq.reshape(-1, N_FEATURES)
synth_flattened = generated_data.reshape(-1, N_FEATURES)

# --- 6. VISUALIZATION ---
# (Visualization Code same as before, ensures output directory exists)
plt.figure(figsize=(14, 6))
plt.plot(data_seq[0, :, 3], label='Real', color='dodgerblue')
plt.plot(generated_data[0, :, 3], label='Synthetic', color='crimson', linestyle='--')
plt.title('Real vs Synthetic Bitcoin Price')
plt.legend()
plt.savefig(os.path.join(OUTPUT_DIR, 'comparison.png'))
plt.show()

# PCA
pca = PCA(n_components=2)
combined = np.concatenate([real_flattened[:2000], synth_flattened[:2000]])
pca_res = pca.fit_transform(combined)
plt.figure(figsize=(10,6))
plt.scatter(pca_res[:2000,0], pca_res[:2000,1], alpha=0.2, label='Real')
plt.scatter(pca_res[2000:,0], pca_res[2000:,1], alpha=0.2, label='Synth', color='red')
plt.legend()
plt.title("PCA Distribution")
plt.savefig(os.path.join(OUTPUT_DIR, 'pca.png'))
plt.show()

print("Graphs saved.")